# Data Fusion: Goodreads, Amazon, DBpedia (Phase III)

**Entity = edition.** Input: the three translated XML files (target schema) and the one-to-one correspondences from Identity Resolution.
Output: **one fused record per real-world edition**, with missing values filled (slot filling) and conflicting values resolved (conflict resolution).

Workflow, following the lecture (WDI06) and the PyDI exercise:
1. Setup, load datasets and correspondences
2. Profile the inputs: density, attribute coverage of the matched records (potential for slot filling)
3. Record groups: group size distribution, suspicious groups caused by matching errors
4. Provenance metadata (trust scores per dataset)
5. Evaluation set for fusion (>= 15 entities, >= 4 attributes, true values from an **external** source)
6. Baseline fusion + consistency, density, accuracy
7. Experiments: conflict resolution functions per attribute, source preference
8. Final strategy, error analysis, summary for the report

Cells marked **TODO (team)** need a decision or manual work from you.

## 1. Setup

In [ ]:
import itertools
import logging
import os
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

os.makedirs("logs", exist_ok=True)
# INFO: the fusion engine then prints density and consistency per attribute (needed for the report)
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", force=True)

import PyDI.fusion as pf
from PyDI.io import load_xml
from PyDI.utils import DataProfiler
from PyDI.fusion import (DataFusionStrategy, DataFusionEngine, DataFusionEvaluator,
                         longest_string, prefer_higher_trust, union, tokenized_match)
from PyDI.fusion.engine import build_record_groups_from_correspondences

DATA_DIR = Path("data")
CORR_DIR = DATA_DIR / "correspondences"
OUT_DIR = DATA_DIR / "output" / "data_fusion"
EVAL_DIR = Path("gold_standard") / "fusion"          # evaluation set for fusion (NOT the identity-resolution gold standard)
for d in (OUT_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

NAMES = ["goodreads", "amazon", "dbpedia"]
PAIRS = [("goodreads", "amazon"), ("goodreads", "dbpedia"), ("amazon", "dbpedia")]
SEED = 42

# attributes of the integrated schema that we fuse (>= 5 required)
STRING_ATTRS = ["title", "publisher", "publication_date", "format", "language"]
NUMERIC_ATTRS = ["page_count", "rating", "ratings_count", "price"]
LIST_ATTRS = ["authors_author"]
FUSE_ATTRS = STRING_ATTRS + LIST_ATTRS + NUMERIC_ATTRS
# attributes for which the truth can be looked up externally (rating, ratings_count, price are time-dependent / subjective)
EVAL_ATTRS = ["title", "authors_author", "publisher", "publication_date", "page_count", "format"]

## 2. Helper functions (pure pandas, no PyDI)

* `is_null`, `to_list`: uniform handling of missing values and list attributes (`authors_author`)
* `tokens`, `similar_values`: the tolerant comparison, same idea as PyDI's `tokenized_match` (lower case, no punctuation, Jaccard on word sets)
* `components`: record groups = connected components of the correspondences (transitive, like the lecture)
* `density`, `consistency`: data-centric evaluation measures (lecture, slides 19-20)
* `attribute_accuracy`: ground-truth-based accuracy per attribute (lecture, slide 21), **strict** and **tolerant**

In [ ]:
def is_null(v):
    if isinstance(v, (list, tuple, set)):
        return len(v) == 0
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except (TypeError, ValueError):
        pass
    return str(v).strip() in ("", "nan", "<NA>", "None")


def to_list(v):
    """authors_author -> list of non-empty strings, or None."""
    if isinstance(v, (list, tuple, set)):
        out = [str(x).strip() for x in v if not is_null(x)]
    elif is_null(v):
        out = []
    else:
        out = [str(v).strip()]
    return out or None


def tokens(v):
    """Set of lower-cased alphanumeric words; lists are flattened (all words of all names)."""
    if isinstance(v, (list, tuple, set)):
        v = " ".join(str(x) for x in v)
    s = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode().lower()
    return set(re.sub(r"[^a-z0-9 ]", " ", s).split())


def tok_jaccard(a, b):
    ta, tb = tokens(a), tokens(b)
    return len(ta & tb) / len(ta | tb) if (ta | tb) else 1.0


def strict_equal(a, b):
    """Exact comparison after trimming / lower-casing; lists compare as sets; numbers numerically."""
    if isinstance(a, (list, tuple)) or isinstance(b, (list, tuple)):
        return {str(x).strip().lower() for x in (a or [])} == {str(x).strip().lower() for x in (b or [])}
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return str(a).strip().lower() == str(b).strip().lower()


def similar_values(a, b, attr, thr=0.8):
    """Tolerant comparison: word-Jaccard >= thr for text/lists, exact for numbers."""
    if attr in NUMERIC_ATTRS:
        return strict_equal(a, b)
    return tok_jaccard(a, b) >= thr


def components(*corr_dfs):
    """Union-find over all correspondences. Returns {record id: group id}."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for c in corr_dfs:
        for a, b in zip(c["id1"], c["id2"]):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    return {x: find(x) for x in list(parent)}


def group_table(corr_dfs, source_of):
    """One row per record group: size and number of records per dataset."""
    comp = components(*corr_dfs)
    members = defaultdict(list)
    for rid, g in comp.items():
        members[g].append(rid)
    rows = []
    for g, ids in members.items():
        cnt = Counter(source_of[i] for i in ids)
        rows.append({"group": g, "size": len(ids), **{n: cnt.get(n, 0) for n in NAMES}, "ids": sorted(ids)})
    t = pd.DataFrame(rows)
    t["clean"] = (t["size"] <= len(NAMES)) & (t[NAMES].max(axis=1) <= 1)   # at most one record per dataset
    return t


def drop_dirty_correspondences(corr_dfs, gt):
    """Remove all correspondences that lie inside a group that is not clean."""
    comp = components(*corr_dfs)
    dirty = set(gt.loc[~gt["clean"], "group"])
    return [c[[comp[a] not in dirty for a in c["id1"]]].reset_index(drop=True) for c in corr_dfs]


def density(df, attrs):
    """Fraction of non-null values per attribute and for the whole table (lecture, slide 19)."""
    col = {a: float((~df[a].map(is_null)).mean()) for a in attrs if a in df.columns}
    col["TABLE"] = float(np.mean([col[a] for a in col]))
    return pd.Series(col)


def consistency(datasets, gt, attrs, thr=None):
    """Share of groups (>= 2 records) without conflicting values per attribute (lecture, slide 20).
    thr=None: values must be identical (after trimming/lower case); thr=0.8: similar values count as equal."""
    by_id = {}
    for df in datasets:
        d = df.set_index("id")
        for a in attrs:
            by_id.setdefault(a, {}).update(d[a].to_dict())
    out = {}
    groups = gt[gt["size"] >= 2]
    for a in attrs:
        ok = 0
        for ids in groups["ids"]:
            vals = [by_id[a][i] for i in ids if i in by_id[a] and not is_null(by_id[a][i])]
            if thr is None:
                good = all(strict_equal(vals[0], v) for v in vals[1:])
            else:
                good = all(similar_values(vals[0], v, a, thr) for v in vals[1:])
            ok += good
        out[a] = ok / max(len(groups), 1)
    s = pd.Series(out)
    s["TABLE"] = s.mean()
    return s


def attach_eval_id(fused, expected_ids):
    """The evaluation set names each entity by the ID of ONE of its records; _fusion_sources lists all fused IDs."""
    out = fused.copy()
    out["eval_id"] = out["_fusion_sources"].apply(lambda s: next((x for x in s if x in expected_ids), None))
    return out


def attribute_accuracy(fused, expected, attrs, thr=0.8):
    """Accuracy per attribute against the evaluation set: strict and tolerant (see similar_values)."""
    f = fused[fused["eval_id"].notna()].drop_duplicates("eval_id").set_index("eval_id")
    ex = expected.set_index("id")
    rows = []
    for a in attrs:
        n = s = t = 0
        for eid, true in ex[a].items() if a in ex.columns else []:
            if is_null(true) or eid not in f.index:
                continue
            got = f.at[eid, a]
            n += 1
            s += (not is_null(got)) and strict_equal(got, true)
            t += (not is_null(got)) and similar_values(got, true, a, thr)
        rows.append({"attribute": a, "values": n, "accuracy strict": s / n if n else np.nan,
                     "accuracy tolerant": t / n if n else np.nan})
    r = pd.DataFrame(rows).set_index("attribute")
    r.loc["OVERALL"] = [r["values"].sum(),
                        (r["accuracy strict"] * r["values"]).sum() / max(r["values"].sum(), 1),
                        (r["accuracy tolerant"] * r["values"]).sum() / max(r["values"].sum(), 1)]
    return r

## 3. Load datasets and correspondences

In [ ]:
def load(name):
    df = load_xml(DATA_DIR / f"{name}.xml", name=name)
    df.attrs["source"] = name
    for c in FUSE_ATTRS:
        if c not in df.columns:
            df[c] = None
    for c in NUMERIC_ATTRS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["authors_author"] = df["authors_author"].map(to_list)
    return df


dfs = {n: load(n) for n in NAMES}
source_of = {i: n for n, df in dfs.items() for i in df["id"]}
assert len(source_of) == sum(len(d) for d in dfs.values()), "record ids must be unique across the datasets"


def load_corr(a, b):
    c = pd.read_csv(CORR_DIR / f"{a}_{b}.csv")
    if "similarity" not in c.columns:                 # the exercise files use the column name 'similarity'
        c["similarity"] = c["score"]
    return c[["id1", "id2", "similarity"]]


corr = {f"{a}_{b}": load_corr(a, b) for a, b in PAIRS}
for n, df in dfs.items():
    print(f"{n:<10} {len(df):>7,} records")
for p, c in corr.items():
    print(f"{p:<20} {len(c):>7,} correspondences")

## 4. Profile the inputs

**Density** of the input datasets and the **potential for slot filling**: only records that have a correspondence can fill gaps in other datasets,
so the coverage table is computed on the matched records only (like Task 1 of the exercise).

In [ ]:
dens = pd.DataFrame({n: density(df, FUSE_ATTRS) for n, df in dfs.items()})
print("=== Density of the input datasets (all records) ===")
display(dens.round(3))

matched_ids = set(pd.concat([c[["id1", "id2"]].stack() for c in corr.values()]))
print("\n=== Records with a correspondence, per dataset ===")
display(pd.Series({n: int(df["id"].isin(matched_ids).sum()) for n, df in dfs.items()}))

matched_dens = pd.DataFrame({n: density(df[df["id"].isin(matched_ids)], FUSE_ATTRS) for n, df in dfs.items()})
print("=== Density of the matched records only (which dataset can fill which attribute?) ===")
display(matched_dens.round(3))

## 5. Record groups and group size distribution

All correspondences of the three pairs are chained: records that (transitively) describe the same edition end up in one group.
Because every source lists an edition at most once and each pair is matched one-to-one, a **valid group has at most 3 records, one per dataset**.
Larger groups, or groups with two records of the same dataset, can only come from wrong correspondences that got chained together.

In [ ]:
corr_list = [corr[f"{a}_{b}"] for a, b in PAIRS]

# PyDI's own grouping (prints the group size distribution, as in the exercise)
record_groups = build_record_groups_from_correspondences(list(dfs.values()), corr_list, id_column="id")

gt = group_table(corr_list, source_of)
print("\n=== Group size distribution (own computation, same grouping) ===")
dist = gt["size"].value_counts().sort_index().rename("groups").to_frame()
dist["share"] = (dist["groups"] / dist["groups"].sum()).round(4)
display(dist)
print(f"groups: {len(gt):,} | clean groups (<= 1 record per dataset): {gt['clean'].sum():,} | "
      f"not clean: {(~gt['clean']).sum():,} ({(~gt['clean']).mean():.1%})")
print("\n=== Composition of the groups (records per dataset) ===")
display(gt.groupby(NAMES).size().rename("groups").reset_index().sort_values("groups", ascending=False).head(12))

In [ ]:
# Look at a few suspicious groups: do the records really describe the same edition?
COLS = ["title", "authors_author", "publisher", "publication_date", "page_count", "format"]
by_id = pd.concat([df.set_index("id")[COLS].assign(dataset=n) for n, df in dfs.items()])
pd.set_option("display.max_colwidth", 45)
for _, g in gt[~gt["clean"]].sort_values("size", ascending=False).head(3).iterrows():
    print(f"\n--- group of size {g['size']} ---")
    display(by_id.loc[g["ids"]])

### Decision: keep or remove suspicious groups? **TODO (team)**

`FILTER_MODE = "none"` fuses every group as it is. `"clean_only"` removes all correspondences inside groups that are not clean; their records stay in the
integrated dataset as unfused singletons. Run both, compare density / consistency / accuracy and write down which one you use and why
(trade-off: fewer wrong merges vs. fewer entities with filled values).

In [ ]:
FILTER_MODE = "clean_only"        # "none" or "clean_only"

if FILTER_MODE == "clean_only":
    corr_list_used = drop_dirty_correspondences(corr_list, gt)
    print("correspondences:", [len(c) for c in corr_list], "->", [len(c) for c in corr_list_used])
else:
    corr_list_used = corr_list
gt_used = group_table(corr_list_used, source_of)
print(f"groups used for fusion: {len(gt_used):,} (size distribution: {gt_used['size'].value_counts().sort_index().to_dict()})")

## 6. Provenance metadata: trust per dataset

`prefer_higher_trust` reads the score from `df.attrs["trust"]`. We start with **equal trust** (no prior knowledge; ties go to the dataset that comes first in the list)
and let the evaluation set tell us which source is reliable for which attribute (section 9), as in the DBpedia case study of the lecture, where the best
function was learned on a ground truth.
Also document the **dataset date** (when the data was collected). **TODO (team):** fill in the real collection dates / your own quality score and explain in the report why you set them this way.

In [ ]:
PRIOR_TRUST = {"goodreads": 1.0, "amazon": 1.0, "dbpedia": 1.0}
DATASET_DATE = {"goodreads": None, "amazon": None, "dbpedia": None}      # TODO (team): collection year of each dataset


def set_trust(trust):
    for n, df in dfs.items():
        df.attrs["trust"] = float(trust[n])
        df.attrs["dataset_date"] = DATASET_DATE[n]


def trust_from_order(order):
    """['amazon','goodreads','dbpedia'] -> amazon 3, goodreads 2, dbpedia 1"""
    return {n: len(order) - i for i, n in enumerate(order)}


set_trust(PRIOR_TRUST)

## 7. Evaluation set for fusion

Requirements (exercise slides): **>= 15 entities (more is better), >= 4 attributes per entity**, true values looked up manually in an **external** source
(not one of our three datasets, e.g. Open Library, Library of Congress, WorldCat, publisher page; ISBNs below make the lookup unambiguous).
It should contain entities that **actually have conflicts**. Identify each entity by the ID of **one** of its records (never two records of the same group).

Because the entities are edition-level, look up the **edition** (ISBN), not the work.

**Step 1** exports candidate groups (clean groups with conflicts) to a CSV. **Step 2** (manual): fill the `TRUE__*` columns, separate several authors by `;`.
**Step 3** turns the filled CSV into the XML evaluation set in the target schema.

In [ ]:
N_CANDIDATES = 60          # export more than you need; skip entities you cannot verify
CAND_PATH = EVAL_DIR / "fusion_eval_candidates.csv"
EVAL_XML = EVAL_DIR / "fusion_evaluation_set.xml"

# optional: ISBNs from the raw files (helps to look up the right edition); needs the raw files from the schema-mapping phase
isbn_lookup = {}
try:
    from build_gold_standard import raw_isbns, xml_titles, isbn13s
    raw = {"goodreads": ("data/books_1.Best_Books_Ever.csv", "goodreads_id_"),
           "amazon": ("data/amazon_books_subset.csv", "amazon_id_"),
           "dbpedia": ("data/dbpedia_book_data.xml", "dbpedia_id_")}
    for n, (path, prefix) in raw.items():
        isbn_lookup.update({k: ", ".join(sorted(isbn13s(v))) for k, v in raw_isbns(n, path, xml_titles(DATA_DIR / f"{n}.xml"), prefix).items()})
except Exception as e:                                      # noqa
    print("No ISBNs available:", repr(e))


def conflicting_attrs(ids):
    n = 0
    for a in EVAL_ATTRS:
        vals = [by_attr[a][i] for i in ids if not is_null(by_attr[a].get(i))]
        n += len(vals) >= 2 and not all(similar_values(vals[0], v, a, 0.999) for v in vals[1:])
    return n


by_attr = {a: {} for a in EVAL_ATTRS}
for df in dfs.values():
    for a in EVAL_ATTRS:
        by_attr[a].update(df.set_index("id")[a].to_dict())

if not CAND_PATH.exists():
    clean = gt_used[(gt_used["clean"]) & (gt_used["size"] >= 2)].copy()
    clean["n_conflicts"] = clean["ids"].map(conflicting_attrs)
    pool = clean[clean["n_conflicts"] >= 2]
    sample = pool.sample(min(N_CANDIDATES, len(pool)), random_state=SEED)
    rows = []
    for _, g in sample.iterrows():
        r = {"eval_id": g["ids"][0], "members": " | ".join(g["ids"]),
             "isbn(s)": " / ".join(filter(None, (isbn_lookup.get(i, "") for i in g["ids"])))}
        for i in g["ids"]:
            for a in EVAL_ATTRS:
                v = by_attr[a].get(i)
                r[f"{a}__{source_of[i]}"] = "; ".join(v) if isinstance(v, list) else v
        r.update({f"TRUE__{a}": "" for a in EVAL_ATTRS})
        rows.append(r)
    pd.DataFrame(rows).to_csv(CAND_PATH, index=False)
    print(f"{len(rows)} candidates written to {CAND_PATH}  ->  fill the TRUE__ columns manually")
else:
    print("candidate file exists:", CAND_PATH)

In [ ]:
def build_eval_xml(filled_csv, xml_path):
    """Filled candidate CSV -> XML evaluation set in the target schema (same structure as our dataset XMLs)."""
    import xml.etree.ElementTree as ET
    d = pd.read_csv(filled_csv, dtype=str)
    root, kept = ET.Element("books"), 0
    for _, r in d.iterrows():
        truth = {a: r.get(f"TRUE__{a}") for a in EVAL_ATTRS if not is_null(r.get(f"TRUE__{a}"))}
        if len(truth) < 4:
            continue                                         # requirement: >= 4 attributes per entity
        b = ET.SubElement(root, "book", id=r["eval_id"])
        for a, v in truth.items():
            if a == "authors_author":
                au = ET.SubElement(b, "authors")
                for name in str(v).split(";"):
                    if name.strip():
                        ET.SubElement(au, "author").text = name.strip()
            else:
                ET.SubElement(b, a).text = str(v).strip()
        kept += 1
    ET.ElementTree(root).write(xml_path, encoding="utf-8", xml_declaration=True)
    print(f"{kept} entities with >= 4 true attributes written to {xml_path}")


FILLED_CSV = EVAL_DIR / "fusion_eval_filled.csv"           # TODO (team): copy of the candidate file with the TRUE__ columns filled
if FILLED_CSV.exists():
    build_eval_xml(FILLED_CSV, EVAL_XML)

In [ ]:
# Load the evaluation set (if it exists) and split it: dev part to choose functions, test part for the final number
df_eval = None
if EVAL_XML.exists():
    df_eval = load_xml(EVAL_XML, name="fusion_gold")
    for c in NUMERIC_ATTRS:
        if c in df_eval.columns:
            df_eval[c] = pd.to_numeric(df_eval[c], errors="coerce")
    if "authors_author" in df_eval.columns:
        df_eval["authors_author"] = df_eval["authors_author"].map(to_list)
    assert set(df_eval["id"]) <= set(source_of), "evaluation IDs must be record IDs of our datasets"
    ids_sorted = sorted(df_eval["id"])
    random.Random(SEED).shuffle(ids_sorted)
    if len(ids_sorted) >= 30:
        half = len(ids_sorted) // 2
        eval_dev_ids, eval_test_ids = set(ids_sorted[:half]), set(ids_sorted[half:])
    else:
        print("WARNING: fewer than 30 entities -> dev and test are identical (results will be optimistic)")
        eval_dev_ids = eval_test_ids = set(ids_sorted)
    df_eval_dev = df_eval[df_eval["id"].isin(eval_dev_ids)].reset_index(drop=True)
    df_eval_test = df_eval[df_eval["id"].isin(eval_test_ids)].reset_index(drop=True)
    expected_ids = set(df_eval["id"])
    print(f"evaluation set: {len(df_eval)} entities (dev {len(df_eval_dev)}, test {len(df_eval_test)}); attributes: "
          f"{[c for c in df_eval.columns if c in FUSE_ATTRS]}")
else:
    expected_ids = set()
    print("No evaluation set yet -> sections that need it are skipped. Create it first (steps 1-3 above).")

## 8. Baseline fusion

Simple functions per attribute (lecture slide 12): longest string for text, union for lists, median for numbers.
The engine prints **density** and **consistency**; copy these numbers into the report. A low consistency means many conflicts, so fusion matters for that attribute.

In [ ]:
def fn(*names):
    """First conflict resolution function of PyDI.fusion that exists under one of the given names."""
    for n in names:
        f = getattr(pf, n, None)
        if f is not None:
            return f
    return None


print("available in PyDI.fusion:", sorted(n for n in dir(pf) if not n.startswith("_") and callable(getattr(pf, n))))

In [ ]:
median_fn = fn("median", "median_value")
BASELINE = {"title": longest_string, "publisher": longest_string, "publication_date": longest_string,
            "format": longest_string, "language": longest_string, "authors_author": union,
            "page_count": median_fn or prefer_higher_trust, "rating": median_fn or prefer_higher_trust,
            "ratings_count": median_fn or prefer_higher_trust, "price": median_fn or prefer_higher_trust}

set_trust(PRIOR_TRUST)
baseline_strategy = DataFusionStrategy("Version_1_Baseline")
for a, f in BASELINE.items():
    baseline_strategy.add_attribute_fuser(a, f)

baseline_engine = DataFusionEngine(baseline_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_debug_baseline.jsonl"))
fused_baseline = baseline_engine.run(datasets=list(dfs.values()), correspondences=corr_list_used,
                                     id_column="id", include_singletons=True)
print(len(fused_baseline), "fused records (including singletons)")
fused_baseline.head(3)

In [ ]:
# Our own consistency / density tables (same definitions as the lecture) for the report
cons_strict = consistency(list(dfs.values()), gt_used, FUSE_ATTRS, thr=None)
cons_tol = consistency(list(dfs.values()), gt_used, FUSE_ATTRS, thr=0.8)
print("=== Consistency of the input groups (share of groups without conflicting values) ===")
display(pd.DataFrame({"identical values": cons_strict, "similar values (>= 0.8)": cons_tol}).round(3))

dens_fused = density(fused_baseline, FUSE_ATTRS)
largest = max(dfs, key=lambda n: len(dfs[n]))
print(f"=== Density: largest input dataset ({largest}) vs fused ===")
display(pd.DataFrame({f"{largest} (input)": dens[largest].reindex(dens_fused.index), "fused": dens_fused}).round(3))

In [ ]:
if df_eval is not None:
    fb = attach_eval_id(fused_baseline, expected_ids)
    evaluator = DataFusionEvaluator(baseline_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_evaluation_debug_baseline.jsonl"),
                                    fusion_debug_logs=str(OUT_DIR / "fusion_debug_baseline.jsonl"))
    m = evaluator.evaluate(fused_df=fb, fused_id_column="eval_id", expected_df=df_eval_dev, expected_id_column="id")
    print("PyDI evaluator, overall accuracy (dev):", round(m["overall_accuracy"], 4))
    print("\n=== Baseline accuracy per attribute (dev), own computation ===")
    display(attribute_accuracy(fb, df_eval_dev, [a for a in EVAL_ATTRS if a in df_eval_dev.columns]).round(3))

## 9. Experiments: which function works for which attribute?

**Source reliability.** For every dataset and attribute: how often is the dataset's own value correct (tolerant comparison, dev part of the evaluation set)?
This is the evidence for the trust scores / source preference (metadata-based conflict resolution).

In [ ]:
if df_eval is not None:
    rows = []
    comp_all = components(*corr_list_used)
    members = defaultdict(list)
    for rid, g in comp_all.items():
        members[g].append(rid)
    for a in [a for a in EVAL_ATTRS if a in df_eval_dev.columns]:
        for n in NAMES:
            ok = tot = 0
            for eid, true in df_eval_dev.set_index("id")[a].items():
                if is_null(true):
                    continue
                rec = [i for i in members.get(comp_all.get(eid, eid), [eid]) if source_of[i] == n]
                if not rec:
                    continue
                val = dfs[n].set_index("id").at[rec[0], a]
                if is_null(val):
                    continue
                tot += 1
                ok += similar_values(val, true, a, 0.8)
            rows.append({"attribute": a, "dataset": n, "values": tot, "correct": ok / tot if tot else np.nan})
    reliability = pd.DataFrame(rows).pivot(index="attribute", columns="dataset", values="correct")[NAMES]
    print("=== Share of correct values per source (only where the source has a value) ===")
    display(reliability.round(2))

**Grid of conflict resolution functions per attribute** (each run fuses only that attribute; for `prefer_higher_trust` all 6 source orders are tried).
Function names are looked up in `PyDI.fusion`; those that do not exist in your PyDI version are skipped (see the list printed above, and the PyDI fusion wiki).
Ties are resolved in favour of the function listed first (simpler first), like in the identity-resolution notebook.

In [ ]:
CANDIDATES = {   # attribute -> function names, simpler first; several names = aliases
    "title":            [("longest_string",), ("shortest_string",), ("voting", "vote"), ("prefer_higher_trust",)],
    "authors_author":   [("union",), ("intersection",), ("intersection_k_sources",), ("prefer_higher_trust",)],
    "publisher":        [("longest_string",), ("voting", "vote"), ("prefer_higher_trust",)],
    "publication_date": [("voting", "vote"), ("most_recent",), ("prefer_higher_trust",)],
    "page_count":       [("median", "median_value"), ("average", "mean"), ("maximum", "max_value"), ("minimum", "min_value"),
                         ("voting", "vote"), ("prefer_higher_trust",)],
    "format":           [("longest_string",), ("voting", "vote"), ("prefer_higher_trust",)],
}
ORDERS = list(itertools.permutations(NAMES))


def run_single(attr, f, order):
    set_trust(trust_from_order(order))
    s = DataFusionStrategy(f"{attr}:{getattr(f, '__name__', 'f')}")
    s.add_attribute_fuser(attr, f)
    lvl = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)
    try:
        out = DataFusionEngine(s).run(datasets=[dfs[n] for n in order], correspondences=corr_list_used, id_column="id")
    finally:
        logging.getLogger().setLevel(lvl)
    return attach_eval_id(out, expected_ids)


grid_rows = []
if df_eval is not None:
    for attr, options in CANDIDATES.items():
        if attr not in df_eval_dev.columns:
            continue
        for names in options:
            f = fn(*names)
            if f is None:
                print(f"skipped (not in PyDI.fusion): {names}")
                continue
            for order in (ORDERS if f is prefer_higher_trust else [tuple(NAMES)]):
                try:
                    acc = attribute_accuracy(run_single(attr, f, order), df_eval_dev, [attr]).loc[attr]
                except Exception as e:                      # keep going, show the problem
                    print(f"failed: {attr} / {names[0]} / {order}: {e!r}")
                    continue
                grid_rows.append({"attribute": attr, "function": names[0],
                                  "trust order": " > ".join(order) if f is prefer_higher_trust else "-",
                                  "strict": acc["accuracy strict"], "tolerant": acc["accuracy tolerant"]})
    grid = pd.DataFrame(grid_rows)
    set_trust(PRIOR_TRUST)
    grid.to_csv(OUT_DIR / "function_grid_dev.csv", index=False)
    for a in grid["attribute"].unique():
        print(f"\n=== {a}: accuracy on the dev part of the evaluation set ===")
        display(grid[grid["attribute"] == a].drop(columns="attribute").sort_values("tolerant", ascending=False).round(3).reset_index(drop=True))

## 10. Final strategy

The best function per attribute (tolerant dev accuracy, ties -> simpler function) is selected automatically; you can override it below.
Because `prefer_higher_trust` uses **one trust score per dataset**, the final engine uses a single source order (`FINAL_ORDER`): take the order that wins for most attributes in the grid.
Numeric attributes without ground truth (`rating`, `ratings_count`, `price`) are fixed by reasoning: ratings are subjective and change over time -> median;
price comes from one source only in most cases -> `prefer_higher_trust`. **TODO (team):** justify these choices in the report.

**Be careful with lenient evaluation:** the threshold 0.8 of the tolerant comparison counts e.g. "Dune" vs "Dune (novel)" as correct. Write down which cases this accepts and why that is fine for your use case.

In [ ]:
FINAL_ORDER = ("amazon", "goodreads", "dbpedia")       # TODO (team): set from the grid / reliability table
OVERRIDE = {}                                           # e.g. {"title": "prefer_higher_trust"}
FIXED = {"rating": ("median", "median_value"), "ratings_count": ("median", "median_value"),
         "price": ("prefer_higher_trust",), "language": ("prefer_higher_trust",)}

chosen = {}
if grid_rows:
    g = pd.DataFrame(grid_rows)
    order_pos = {names[0]: i for opts in CANDIDATES.values() for i, names in enumerate(opts)}
    g["pos"] = g["function"].map(order_pos)
    for a, sub in g.groupby("attribute"):
        chosen[a] = sub.sort_values(["tolerant", "pos"], ascending=[False, True]).iloc[0]["function"]
chosen.update(OVERRIDE)
print("selected per attribute:", chosen)

set_trust(trust_from_order(FINAL_ORDER))
final_strategy = DataFusionStrategy("Version_Final")
for a in FUSE_ATTRS:
    name = chosen.get(a)
    f = fn(name) if name else fn(*FIXED.get(a, ("prefer_higher_trust",)))
    final_strategy.add_attribute_fuser(a, f or prefer_higher_trust)
for a in ["title", "authors_author", "publisher", "format"]:
    final_strategy.add_evaluation_function(a, tokenized_match, threshold=0.8)

final_engine = DataFusionEngine(final_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_debug_final.jsonl"))
fused_final = final_engine.run(datasets=[dfs[n] for n in FINAL_ORDER], correspondences=corr_list_used,
                               id_column="id", include_singletons=True)
fused_final.to_csv(OUT_DIR / "fused_books.csv", index=False)
print(len(fused_final), "records in the integrated dataset ->", OUT_DIR / "fused_books.csv")

In [ ]:
if df_eval is not None:
    ff = attach_eval_id(fused_final, expected_ids)
    evaluator = DataFusionEvaluator(final_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_evaluation_debug_final.jsonl"),
                                    fusion_debug_logs=str(OUT_DIR / "fusion_debug_final.jsonl"))
    attrs_eval = [a for a in EVAL_ATTRS if a in df_eval.columns]
    final_metrics = evaluator.evaluate(fused_df=ff, fused_id_column="eval_id", expected_df=df_eval_test, expected_id_column="id")
    print("PyDI evaluator, overall accuracy on the TEST part:", round(final_metrics["overall_accuracy"], 4))
    cmp = pd.concat({"baseline (test)": attribute_accuracy(attach_eval_id(fused_baseline, expected_ids), df_eval_test, attrs_eval),
                     "final (test)": attribute_accuracy(ff, df_eval_test, attrs_eval)}, axis=1)
    print("\n=== Accuracy per attribute on the TEST part (use this table in the report) ===")
    display(cmp.round(3))

## 11. Error analysis

For every value that is still wrong (tolerant comparison, whole evaluation set): the fused value, the true value, the value of each source,
and whether the **true value was available in any source at all**. If not, no conflict resolution function that chooses among the input values can get it right.

In [ ]:
if df_eval is not None:
    ex = df_eval.set_index("id")
    f = ff[ff["eval_id"].notna()].drop_duplicates("eval_id").set_index("eval_id")
    comp_all = components(*corr_list_used)
    members = defaultdict(list)
    for rid, g in comp_all.items():
        members[g].append(rid)
    rows = []
    for eid in ex.index:
        for a in attrs_eval:
            true, got = ex.at[eid, a], f.at[eid, a] if eid in f.index else None
            if is_null(true) or (not is_null(got) and similar_values(got, true, a, 0.8)):
                continue
            recs = members.get(comp_all.get(eid, eid), [eid])
            src = {source_of[i]: dfs[source_of[i]].set_index("id").at[i, a] for i in recs}
            rows.append({"entity": eid, "attribute": a, "true": true, "fused": got, **src,
                         "true value in a source": any((not is_null(v)) and similar_values(v, true, a, 0.8) for v in src.values())})
    errors = pd.DataFrame(rows)
    print(f"{len(errors)} wrong or missing values; true value available in a source for {int(errors['true value in a source'].sum()) if len(errors) else 0}")
    pd.set_option("display.max_colwidth", 40)
    display(errors.groupby("attribute")["true value in a source"].agg(wrong="size", fixable="sum") if len(errors) else errors)
    display(errors.head(25))

## 12. Summary for the report

The three final questions of the exercise (largest input dataset as reference):
1. How many additional entities compared to the largest input dataset?
2. How much did the density increase compared to the largest input dataset?
3. What is the overall accuracy of the final dataset according to the evaluation set?

In [ ]:
largest = max(dfs, key=lambda n: len(dfs[n]))
n_fused, n_largest = len(fused_final), len(dfs[largest])
d_in, d_out = density(dfs[largest], FUSE_ATTRS)["TABLE"], density(fused_final, FUSE_ATTRS)["TABLE"]
print(f"largest input dataset: {largest} with {n_largest:,} records")
print(f"1. entities in the integrated dataset: {n_fused:,}  ->  {n_fused - n_largest:,} additional entities (+{(n_fused - n_largest) / n_largest:.1%})")
print(f"2. table density: {d_in:.3f} -> {d_out:.3f}  (+{d_out - d_in:.3f})")
if df_eval is not None:
    acc = attribute_accuracy(ff, df_eval_test, attrs_eval).loc["OVERALL"]
    print(f"3. overall accuracy on the test part of the evaluation set: strict {acc['accuracy strict']:.3f}, tolerant {acc['accuracy tolerant']:.3f}")
else:
    print("3. overall accuracy: create the evaluation set first")
print("\nfused records that merge >= 2 source records:", int(fused_final['_fusion_sources'].map(len).ge(2).sum()))

### Report checklist (Data Fusion part, exercise slide 16)
* provenance data you added and why you set the values as you did (section 6)
* density of inputs and fused data (sections 4 and 8), consistency per attribute (section 8)
* size, content and creation of the evaluation set, external source and why you chose it (section 7)
* functions tried per attribute, own functions if any, accuracy table with the best function per attribute (sections 9, 10)
* analysis of the remaining errors, e.g. "the true value is in none of the datasets" (section 11)
* effect of `FILTER_MODE` (section 5) and the strict vs. tolerant evaluation
* final three numbers (section 12)